“Support Ticket Classification & Prioritization using Machine Learning”

Problem Statement 
This project predicts support ticket category and priority using machine learning.
It helps businesses automate ticket handling and improve response time.

In [1]:
#step 01 : in this step i uploaded my dataset.
#without data the model cannot work

import zipfile

with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [2]:
#step 02:extract zipfile
#our dataset iis downloaded in a zip file so i extracted it to csv fi

import zipfile

with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
   zip_ref.extractall()

In [3]:
#step 03: checking the files
#this step shows that all files in colab which helps me to find the correct name from dataset

import os
os.listdir()

['.git', 'customer_support_tickets.csv', 'dataset.zip', 'Task_02.ipynb']

In [4]:
#step 04:load dataset
#in this step i loaded the dataset using pandas
#the pandas is python library used to work with data

import pandas as pd

df = pd.read_csv("customer_support_tickets.csv")
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [5]:
#step 05: select columns
#the dataset has many columns so i only selected which are required columns

df = df[['Ticket Description', 'Ticket Type', 'Ticket Priority']]

df.columns = ['ticket_text', 'category', 'priority']

In [6]:
#step 06:clean the text
#in this stepi converted the text to lowercase which removes special character
#and it improves the performance of model

import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['clean_text'] = df['ticket_text'].apply(clean_text)

In [7]:
#step 07: convert the text into numbers(TF-IDF)
#the ml models cannot understand the text directly so TF-IDF helps to convert the text into numerical form where it give importance to important words


from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_text'])

In [8]:
#step 08:training the category model
#in this step it split data into training and testing
#the train model using logistic regression
#it measure accuracy
#the logistic regression is a machine learning algorithm used for classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, df['category'], test_size=0.2)

model_cat = LogisticRegression()
model_cat.fit(X_train, y_train)

y_pred_cat = model_cat.predict(X_test)

print("Category Accuracy:", accuracy_score(y_test, y_pred_cat))

Category Accuracy: 0.21133412042502953


In [9]:
#step 09:training the priority model
#in this it is same process as training category model but we train for priority
#in this the model predicts high/medium/low

X_train, X_test, y_train, y_test = train_test_split(X, df['priority'], test_size=0.2)

model_pri = LogisticRegression()
model_pri.fit(X_train, y_train)

y_pred_pri = model_pri.predict(X_test)

print("Priority Accuracy:", accuracy_score(y_test, y_pred_pri))

Priority Accuracy: 0.23258559622195984


In [10]:
#step 10:prediction
#this is final step where it takes new ticket as input
#it cleans text and converts to numbers and predicts the category and priority
def predict_ticket(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])

    category = model_cat.predict(vec)[0]
    priority = model_pri.predict(vec)[0]

    return category, priority

print(predict_ticket("payment failed urgent help"))
print(predict_ticket("how to update my account"))

('Product inquiry', 'Low')
('Product inquiry', 'Medium')
